# Consolidando planilhas do Excel com Python

#### Importando Bibliotecas Necessárias

In [ ]:
import re
import pandas as pd
import openpyxl as op

#### Inserindo Função de Consolidação

In [ ]:
def consolidar_cw_por_semana(
    arquivo_entrada: str,
    arquivo_saida: str = "consolidado.xlsx",
    padrao_abas: str = r"^CW(\d+)$",
    manter_nome_aba: bool = False,
    **read_excel_kwargs
) -> pd.DataFrame:
    """
    Consolida abas do tipo CW1, CW2, ... em uma única tabela
    e adiciona a coluna 'semana' com o número extraído do nome da aba.

    Parâmetros:
      - arquivo_entrada: caminho do .xlsx de origem
      - arquivo_saida: caminho do .xlsx de saída
      - padrao_abas: regex para identificar abas válidas (padrão: CW + número)
      - manter_nome_aba: se True, adiciona coluna auxiliar 'aba_origem'
      - read_excel_kwargs: repassa opções do pandas.read_excel (ex: header=0, skiprows=1)

    Retorna:
      - DataFrame consolidado
    """
    xls = pd.ExcelFile(arquivo_entrada, engine="openpyxl")
    regex = re.compile(padrao_abas, re.IGNORECASE)

    dfs = []
    for aba in xls.sheet_names:
        nome = aba.strip()
        m = regex.match(nome)
        if not m:
            continue  # ignora abas que não sejam CW + número

        semana = int(m.group(1))

        df = pd.read_excel(
            arquivo_entrada,
            sheet_name=aba,
            engine="openpyxl",
            **read_excel_kwargs
        )

        df["semana"] = semana
        if manter_nome_aba:
            df["aba_origem"] = nome

        dfs.append(df)

    if not dfs:
        raise ValueError("Nenhuma aba no padrão CW<N> foi encontrada no arquivo.")

    consolidado = pd.concat(dfs, ignore_index=True, sort=False)

    # Exporta
    consolidado.to_excel(arquivo_saida, index=False, engine="openpyxl")
    return consolidado

#### Executando Código

In [ ]:

if __name__ == "__main__":
    # Ajuste aqui os caminhos
    entrada = r".\data\Backlog 2025.xlsx"
    saida = r".\data\Backlog 2025_consolidado.xlsx"

    df_final = consolidar_cw_por_semana(
        arquivo_entrada=entrada,
        arquivo_saida=saida,
        manter_nome_aba=True,   # opcional: ajuda em auditoria
        # Exemplos opcionais se seu arquivo precisar:
        # header=0,
        # skiprows=0,
    )

    print("Consolidação concluída.")
